# LokaSense Inference Notebook

This notebook is the lightweight companion to the training notebook.
It loads the saved checkpoints, shows the latest readiness status, runs local inference, and resolves extracted entities into usable coordinates where possible.


## What This Notebook Does

- Loads the latest local signal and NER checkpoints
- Displays the latest production readiness summary if one exists
- Runs signal classification on editable Indonesian inputs
- Extracts and resolves location-like entities
- Shows the latest scored opportunity groups if the training notebook already generated them


In [1]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from IPython.display import display
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification, AutoTokenizer

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "models").exists(), "Please run this notebook from the repo root."
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SPATIAL_MODELLING_DIR = REPO_ROOT / "04_spatial_engine" / "modelling"
if str(SPATIAL_MODELLING_DIR) not in sys.path:
    sys.path.insert(0, str(SPATIAL_MODELLING_DIR))

import scoring as spatial_scoring

from common.location_resolution import LocationResolver

load_dotenv(REPO_ROOT / ".env")

SIGNAL_MODEL_DIR = REPO_ROOT / "models" / "signal_base"
NER_MODEL_DIR = REPO_ROOT / "models" / "ner_base"
READINESS_FILE = REPO_ROOT / "logs" / "production_readiness.json"
resolver = LocationResolver()
SIGNAL_LABELS = [
    "NEUTRAL",
    "DEMAND_UNMET",
    "DEMAND_PRESENT",
    "SUPPLY_SIGNAL",
    "COMPETITION_HIGH",
    "COMPLAINT",
    "TREND",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert SIGNAL_MODEL_DIR.exists(), f"Missing signal model at {SIGNAL_MODEL_DIR}"
assert NER_MODEL_DIR.exists(), f"Missing NER model at {NER_MODEL_DIR}"

signal_tokenizer = AutoTokenizer.from_pretrained(str(SIGNAL_MODEL_DIR))
signal_model = AutoModelForSequenceClassification.from_pretrained(str(SIGNAL_MODEL_DIR)).to(device)
signal_model.eval()

ner_tokenizer = AutoTokenizer.from_pretrained(str(NER_MODEL_DIR))
ner_model = AutoModelForTokenClassification.from_pretrained(str(NER_MODEL_DIR)).to(device)
ner_model.eval()

display(
    pd.DataFrame(
        [
            {
                "signal_model": str(SIGNAL_MODEL_DIR),
                "ner_model": str(NER_MODEL_DIR),
                "device": str(device),
                "readiness_file": str(READINESS_FILE if READINESS_FILE.exists() else ""),
            }
        ]
    )
)


def regex_tokens(text: str):
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def predict_signal(texts):
    encoded = signal_tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.no_grad():
        logits = signal_model(**encoded).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

    rows = []
    for text, prob in zip(texts, probs):
        ranked = np.argsort(prob)[::-1]
        rows.append(
            {
                "text": text,
                "prediction": SIGNAL_LABELS[int(ranked[0])],
                "confidence": round(float(prob[ranked[0]]), 4),
                "runner_up": SIGNAL_LABELS[int(ranked[1])],
                "runner_up_confidence": round(float(prob[ranked[1]]), 4),
                "third_choice": SIGNAL_LABELS[int(ranked[2])],
                "third_choice_confidence": round(float(prob[ranked[2]]), 4),
            }
        )
    return pd.DataFrame(rows)


def predict_ner_entities(text: str) -> list[dict[str, str]]:
    tokens = regex_tokens(text)
    encoded = ner_tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=128, return_tensors="pt")
    word_ids = encoded.word_ids(batch_index=0)
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.no_grad():
        logits = ner_model(**encoded).logits
    pred_ids = torch.argmax(logits, dim=-1)[0].cpu().tolist()

    aligned = []
    previous_word_id = None
    for idx, word_id in enumerate(word_ids):
        if word_id is None or word_id == previous_word_id:
            previous_word_id = word_id
            continue
        aligned.append((tokens[word_id], ner_model.config.id2label[pred_ids[idx]]))
        previous_word_id = word_id

    entities = []
    current_tokens = []
    current_label = None
    for token, tag in aligned:
        if tag.startswith("B-"):
            if current_tokens:
                entities.append({"entity": " ".join(current_tokens), "label": current_label})
            current_tokens = [token]
            current_label = tag[2:]
        elif tag.startswith("I-") and current_tokens and current_label == tag[2:]:
            current_tokens.append(token)
        else:
            if current_tokens:
                entities.append({"entity": " ".join(current_tokens), "label": current_label})
            current_tokens = []
            current_label = None
    if current_tokens:
        entities.append({"entity": " ".join(current_tokens), "label": current_label})
    return entities


def resolve_predicted_entities(text: str, city_hint: str = "", area_hint: str = "") -> pd.DataFrame:
    entities = predict_ner_entities(text)
    resolved = resolver.resolve_entities(entities, city_hint=city_hint, area_hint=area_hint)
    if not resolved:
        return pd.DataFrame([{"entity": "", "label": "NO_ENTITY", "resolved_city": city_hint, "resolved_area": area_hint, "lat": None, "lng": None, "resolution_source": "unresolved"}])
    return pd.DataFrame(resolved)


/home/parasite/Project/Competition/UGM_HACKATHON/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,signal_model,ner_model,device,readiness_file
0,/home/parasite/Project/Competition/UGM_HACKATH...,/home/parasite/Project/Competition/UGM_HACKATH...,cuda,/home/parasite/Project/Competition/UGM_HACKATH...


The setup cell fails early if a checkpoint is missing, which keeps the notebook honest about whether the training path has been run.


In [2]:
if READINESS_FILE.exists():
    with open(READINESS_FILE, "r", encoding="utf-8") as handle:
        readiness_payload = json.load(handle)
    display(pd.DataFrame(readiness_payload["checks"]))
    display(pd.DataFrame([{"production_status": readiness_payload["status"], "failed_checks": ", ".join(readiness_payload.get("failed_checks", []))}]))
else:
    print("No production readiness file found yet. Run training.ipynb first.")


,check,passed,details
0,free_by_default,True,Notebook did not run Google Maps refresh.
1,train_test_leakage_zero,True,0
2,signal_train_rows,True,1105
3,signal_macro_f1,True,0.718838
4,signal_gold_set_present,True,70
5,signal_gold_macro_f1,True,0.927789
6,ner_micro_f1,True,0.791908
7,opportunity_groups,True,24
8,all_signal_classes_seen_in_train,True,7


,production_status,failed_checks
0,ready,


This is the fastest way to see whether the latest notebook run actually cleared the production gates.


In [3]:
inference_inputs = pd.DataFrame(
    [
        {
            "text": "tolong buka kedai kopi murah di lowokwaru malang, anak kos butuh tempat nongkrong dekat kampus",
            "city": "Malang",
            "area_hint": "Lowokwaru",
            "business_hint": "kedai kopi",
        },
        {
            "text": "cafe di wonokromo sekarang saingannya banyak banget, hampir tiap jalan ada tempat baru",
            "city": "Surabaya",
            "area_hint": "Wonokromo",
            "business_hint": "cafe",
        },
        {
            "text": "bakso ini lagi viral di kotagede jogja, rame terus dan reviewnya bagus",
            "city": "Yogyakarta",
            "area_hint": "Kotagede",
            "business_hint": "bakso",
        },
        {
            "text": "pelayanan warung makan di cicendo mengecewakan, mahal dan rasanya biasa aja",
            "city": "Bandung",
            "area_hint": "Cicendo",
            "business_hint": "warung makan",
        },
    ]
)
display(inference_inputs)


,text,city,area_hint,business_hint
0,tolong buka kedai kopi murah di lowokwaru mala...,Malang,Lowokwaru,kedai kopi
1,cafe di wonokromo sekarang saingannya banyak b...,Surabaya,Wonokromo,cafe
2,"bakso ini lagi viral di kotagede jogja, rame t...",Yogyakarta,Kotagede,bakso
3,pelayanan warung makan di cicendo mengecewakan...,Bandung,Cicendo,warung makan


Edit this cell whenever you want to test your own Indonesian text.
The city and area hints are optional, but they help location resolution downstream.


In [4]:
signal_predictions = predict_signal(inference_inputs["text"].tolist())
signal_results = pd.concat([inference_inputs, signal_predictions.drop(columns=["text"])], axis=1)
display(signal_results)


,text,city,area_hint,business_hint,prediction,confidence,runner_up,runner_up_confidence,third_choice,third_choice_confidence
0,tolong buka kedai kopi murah di lowokwaru mala...,Malang,Lowokwaru,kedai kopi,SUPPLY_SIGNAL,0.9622,DEMAND_UNMET,0.0206,NEUTRAL,0.0082
1,cafe di wonokromo sekarang saingannya banyak b...,Surabaya,Wonokromo,cafe,TREND,0.5320,NEUTRAL,0.3837,COMPETITION_HIGH,0.0502
2,"bakso ini lagi viral di kotagede jogja, rame t...",Yogyakarta,Kotagede,bakso,TREND,0.9933,COMPLAINT,0.0023,DEMAND_UNMET,0.0014
3,pelayanan warung makan di cicendo mengecewakan...,Bandung,Cicendo,warung makan,COMPLAINT,0.9942,TREND,0.0013,DEMAND_PRESENT,0.0011


The signal output exposes the top label plus the next two alternatives so uncertainty stays visible.


In [5]:
resolved_outputs = []
for _, row in inference_inputs.iterrows():
    resolved = resolve_predicted_entities(row["text"], city_hint=row["city"], area_hint=row["area_hint"])
    resolved_outputs.append(
        {
            "text": row["text"],
            "resolved_entities": resolved.to_dict(orient="records"),
        }
    )

resolved_outputs_df = pd.DataFrame(resolved_outputs)
display(resolved_outputs_df)


,text,resolved_entities
0,tolong buka kedai kopi murah di lowokwaru mala...,"[{'entity': 'kopi', 'label': 'ORG', 'resolved_..."
1,cafe di wonokromo sekarang saingannya banyak b...,"[{'entity': 'cafe', 'label': 'ORG', 'resolved_..."
2,"bakso ini lagi viral di kotagede jogja, rame t...","[{'entity': 'bakso', 'label': 'ORG', 'resolved..."
3,pelayanan warung makan di cicendo mengecewakan...,"[{'entity': 'warung makan', 'label': 'ORG', 'r..."


The NER output is resolved into city, area, and coordinate hints where the gazetteer can support it.
That closes the gap between raw entity extraction and spatial aggregation.


In [6]:
latest_scores_path = REPO_ROOT / "logs" / "opportunity_scores.csv"
if latest_scores_path.exists():
    latest_scores_df = pd.read_csv(latest_scores_path)
    if not latest_scores_df.empty and "opportunity_score" in latest_scores_df.columns:
        display(latest_scores_df.sort_values("opportunity_score", ascending=False).head(20))
    else:
        print("Opportunity scoring output exists but is still empty.")
else:
    print("No opportunity scoring output found yet. Run training.ipynb first.")


,city,kecamatan,business_type,opportunity_score,color,label,total_signals,signal_breakdown,raw_signal_score,raw_score_after_penalty,franchise_ratio,avg_age_days,avg_decay_weight,resolved_lat,resolved_lng,resolution_source
11,Semarang,Gajahmungkur,ayam bakar,1.000,green,Strong Opportunity,5,{'DEMAND_UNMET': 5},0.3000,0.3000,0.0,178.69,0.0967,-7.009682,110.404685,area_centroid
2,Bandung,Lengkong,kopi,0.981,green,Strong Opportunity,16,"{'DEMAND_UNMET': 15, 'TREND': 1}",0.2884,0.2884,0.0,175.65,0.2128,-6.928765,107.619587,area_centroid
19,Yogyakarta,Gondokusuman,warung makan,0.973,green,Strong Opportunity,17,"{'DEMAND_UNMET': 14, 'SUPPLY_SIGNAL': 2, 'TREN...",0.2840,0.2840,0.0,273.70,0.0731,-7.785633,110.382189,area_centroid
0,Bandung,Cicendo,kopi,0.928,green,Strong Opportunity,16,"{'DEMAND_UNMET': 11, 'TREND': 3, 'DEMAND_PRESE...",0.2568,0.2568,0.0,51.57,0.5529,-6.905082,107.592185,area_centroid
12,Semarang,Gajahmungkur,kuliner,0.764,green,Strong Opportunity,12,"{'DEMAND_UNMET': 8, 'DEMAND_PRESENT': 2, 'NEUT...",0.1584,0.1584,0.0,127.75,0.2692,-7.009682,110.404685,area_centroid
23,Yogyakarta,Umbulharjo,mie,0.728,green,Strong Opportunity,18,"{'NEUTRAL': 9, 'DEMAND_UNMET': 3, 'TREND': 3, ...",0.1366,0.1366,0.0,217.50,0.1249,-7.813068,110.385494,area_centroid
13,Surabaya,Genteng,soto,0.697,green,Strong Opportunity,16,"{'TREND': 13, 'DEMAND_PRESENT': 1, 'COMPETITIO...",0.1181,0.1181,0.0,127.41,0.4237,-7.260760,112.744467,area_centroid
7,Malang,Klojen,nasi padang,0.671,green,Strong Opportunity,6,"{'DEMAND_PRESENT': 3, 'TREND': 2, 'SUPPLY_SIGN...",0.1028,0.1028,0.0,88.63,0.3696,-7.973856,112.627379,area_centroid
5,Malang,Kedungkandang,ayam bakar,0.671,green,Strong Opportunity,13,"{'DEMAND_PRESENT': 7, 'COMPETITION_HIGH': 4, '...",0.1027,0.1027,0.0,78.68,0.3634,-7.988064,112.654722,area_centroid
1,Bandung,Lengkong,es teh,0.667,green,Strong Opportunity,11,{'TREND': 11},0.1000,0.1000,0.0,145.82,0.2183,-6.928765,107.619587,area_centroid


This last cell gives you the latest scored market view without retraining anything.
